# Data Cleaning for ProMatch

In this notebook I take the raw FIFA 22 player data and clean it so my app can use it.
I keep only the columns I need, rename a few of them, remove goalkeepers, and remove any
players that are missing attribute values. At the end I save a smaller file called
`players.csv`, which is the file my app actually loads into the database.

## 1. Read the dataset

I use pandas to read the raw csv file into a table (a DataFrame). `df.shape` tells me how
many rows (players) and columns it has.

In [1]:
import pandas as pd

df = pd.read_csv("players_22_full.csv")
print("rows and columns:", df.shape)
df.head()

rows and columns: (19239, 110)


/var/folders/hc/hk1fn3mj6px3gw3n3jdt_md00000gn/T/ipykernel_27223/4229624518.py:3: DtypeWarning: Columns (0: nation_position, 1: nation_logo_url) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("players_22_full.csv")


,sofifa_id,player_url,short_name,long_name,player_positions,overall,potential,value_eur,wage_eur,age,...,lcb,cb,rcb,rb,gk,player_face_url,club_logo_url,club_flag_url,nation_logo_url,nation_flag_url
0,158023,https://sofifa.com/player/158023/lionel-messi/...,L. Messi,Lionel Andrés Messi Cuccittini,"RW, ST, CF",93,93,78000000.0,320000.0,34,...,50+3,50+3,50+3,61+3,19+3,https://cdn.sofifa.net/players/158/023/22_120.png,https://cdn.sofifa.net/teams/73/60.png,https://cdn.sofifa.net/flags/fr.png,https://cdn.sofifa.net/teams/1369/60.png,https://cdn.sofifa.net/flags/ar.png
1,188545,https://sofifa.com/player/188545/robert-lewand...,R. Lewandowski,Robert Lewandowski,ST,92,92,119500000.0,270000.0,32,...,60+3,60+3,60+3,61+3,19+3,https://cdn.sofifa.net/players/188/545/22_120.png,https://cdn.sofifa.net/teams/21/60.png,https://cdn.sofifa.net/flags/de.png,https://cdn.sofifa.net/teams/1353/60.png,https://cdn.sofifa.net/flags/pl.png
2,20801,https://sofifa.com/player/20801/c-ronaldo-dos-...,Cristiano Ronaldo,Cristiano Ronaldo dos Santos Aveiro,"ST, LW",91,91,45000000.0,270000.0,36,...,53+3,53+3,53+3,60+3,20+3,https://cdn.sofifa.net/players/020/801/22_120.png,https://cdn.sofifa.net/teams/11/60.png,https://cdn.sofifa.net/flags/gb-eng.png,https://cdn.sofifa.net/teams/1354/60.png,https://cdn.sofifa.net/flags/pt.png
3,190871,https://sofifa.com/player/190871/neymar-da-sil...,Neymar Jr,Neymar da Silva Santos Júnior,"LW, CAM",91,91,129000000.0,270000.0,29,...,50+3,50+3,50+3,62+3,20+3,https://cdn.sofifa.net/players/190/871/22_120.png,https://cdn.sofifa.net/teams/73/60.png,https://cdn.sofifa.net/flags/fr.png,NaN,https://cdn.sofifa.net/flags/br.png
4,192985,https://sofifa.com/player/192985/kevin-de-bruy...,K. De Bruyne,Kevin De Bruyne,"CM, CAM",91,91,125500000.0,350000.0,30,...,69+3,69+3,69+3,75+3,21+3,https://cdn.sofifa.net/players/192/985/22_120.png,https://cdn.sofifa.net/teams/10/60.png,https://cdn.sofifa.net/flags/gb-eng.png,https://cdn.sofifa.net/teams/1325/60.png,https://cdn.sofifa.net/flags/be.png


## 2. Keep only the columns I need

The raw file has 110 columns but I only need a few: the player's name, club, nationality,
age, overall rating, positions, and the six attributes I match players on.

In [2]:
columns_i_need = [
    "short_name",
    "club_name",
    "nationality_name",
    "age",
    "overall",
    "player_positions",
    "pace",
    "shooting",
    "passing",
    "dribbling",
    "defending",
    "physic",
]

df = df[columns_i_need]
print("rows and columns:", df.shape)
df.head()

rows and columns: (19239, 12)


,short_name,club_name,nationality_name,age,overall,player_positions,pace,shooting,passing,dribbling,defending,physic
0,L. Messi,Paris Saint-Germain,Argentina,34,93,"RW, ST, CF",85.0,92.0,91.0,95.0,34.0,65.0
1,R. Lewandowski,FC Bayern München,Poland,32,92,ST,78.0,92.0,79.0,86.0,44.0,82.0
2,Cristiano Ronaldo,Manchester United,Portugal,36,91,"ST, LW",87.0,94.0,80.0,88.0,34.0,75.0
3,Neymar Jr,Paris Saint-Germain,Brazil,29,91,"LW, CAM",91.0,83.0,86.0,94.0,37.0,63.0
4,K. De Bruyne,Manchester City,Belgium,30,91,"CM, CAM",76.0,86.0,93.0,88.0,64.0,78.0


## 3. Rename some columns to simpler names

A few column names are a bit odd (like `short_name` and `physic`), so I rename them to
clearer names that I will use in the rest of the project.

In [3]:
df = df.rename(columns={
    "short_name": "name",
    "club_name": "club",
    "nationality_name": "nationality",
    "player_positions": "positions",
    "physic": "physical",
})
df.head()

,name,club,nationality,age,overall,positions,pace,shooting,passing,dribbling,defending,physical
0,L. Messi,Paris Saint-Germain,Argentina,34,93,"RW, ST, CF",85.0,92.0,91.0,95.0,34.0,65.0
1,R. Lewandowski,FC Bayern München,Poland,32,92,ST,78.0,92.0,79.0,86.0,44.0,82.0
2,Cristiano Ronaldo,Manchester United,Portugal,36,91,"ST, LW",87.0,94.0,80.0,88.0,34.0,75.0
3,Neymar Jr,Paris Saint-Germain,Brazil,29,91,"LW, CAM",91.0,83.0,86.0,94.0,37.0,63.0
4,K. De Bruyne,Manchester City,Belgium,30,91,"CM, CAM",76.0,86.0,93.0,88.0,64.0,78.0


## 4. Remove goalkeepers

Goalkeepers are rated on different skills, so their pace/shooting/etc are empty. My app
only matches outfield players, so I remove anyone whose position is goalkeeper (GK).

In [4]:
df = df[~df["positions"].str.contains("GK", na=False)]
print("rows and columns after removing goalkeepers:", df.shape)

rows and columns after removing goalkeepers: (17107, 12)


## 5. Remove players with missing attribute values

Just in case any outfield player is still missing one of the six attributes, I drop those
rows so every player has all six numbers (otherwise the matching maths would break later).

In [5]:
attributes = ["pace", "shooting", "passing", "dribbling", "defending", "physical"]
df = df.dropna(subset=attributes)
print("rows and columns after dropping missing values:", df.shape)

rows and columns after dropping missing values: (17107, 12)


## 6. Make the number columns whole numbers

The attributes are whole numbers (like 85), so I convert them from decimals to integers to
keep the file tidy.

In [6]:
number_columns = ["age", "overall"] + attributes
df[number_columns] = df[number_columns].astype(int)
df.head()

,name,club,nationality,age,overall,positions,pace,shooting,passing,dribbling,defending,physical
0,L. Messi,Paris Saint-Germain,Argentina,34,93,"RW, ST, CF",85,92,91,95,34,65
1,R. Lewandowski,FC Bayern München,Poland,32,92,ST,78,92,79,86,44,82
2,Cristiano Ronaldo,Manchester United,Portugal,36,91,"ST, LW",87,94,80,88,34,75
3,Neymar Jr,Paris Saint-Germain,Brazil,29,91,"LW, CAM",91,83,86,94,37,63
4,K. De Bruyne,Manchester City,Belgium,30,91,"CM, CAM",76,86,93,88,64,78


## 7. Save the cleaned file

Finally I save the cleaned table to `players.csv`. This smaller file is the one my app
loads into the database in the next steps.

In [7]:
df.to_csv("players.csv", index=False)
print("saved players.csv with", df.shape[0], "players")

saved players.csv with 17107 players
